
# Dual-epoch star formation: old and recent bursts leave distinct SED signatures

A galaxy with two separated bursts—one at 10 Gyr (old) and one at 0.3 Gyr (recent)—
produces a SED that blends young hot and old cool stellar populations. Left panel
shows the optical-to-NIR region in linear scale; right panel shows the full
panchromatic SED in log-log, revealing the emission from both young and old stars.


In [ ]:
import warnings

import jax
import matplotlib.pyplot as plt
import numpy as np

import tengri
from tengri.analysis.plotting import setup_style

setup_style()
warnings.filterwarnings("ignore", message=".*BakedInBackend.*")

ssp = tengri.load_ssp()

# Old burst only (10 Gyr ago)
model_old = tengri.SEDModel.build(
    ssp,
    sfh={
        "type": "tsnorm",
        "*": tengri.FIXED,
        "log_peak_sfr": 1.0,
        "peak_lbt_gyr": 10.0,
        "width_gyr": 0.5,
        "skew": 0.1,
        "trunc": 2.0,
    },
    dust={"type": "two_component", "*": tengri.FIXED, "tau_diff": 0.15, "tau_bc": 0.2},
    redshift=tengri.Fixed(0.1),
)

# Recent burst only (0.3 Gyr ago)
model_recent = tengri.SEDModel.build(
    ssp,
    sfh={
        "type": "tsnorm",
        "*": tengri.FIXED,
        "log_peak_sfr": 1.0,
        "peak_lbt_gyr": 0.3,
        "width_gyr": 0.5,
        "skew": 0.3,
        "trunc": 2.0,
    },
    dust={"type": "two_component", "*": tengri.FIXED, "tau_diff": 0.15, "tau_bc": 0.2},
    redshift=tengri.Fixed(0.1),
)

# Double burst: broad peak at intermediate time
model_double = tengri.SEDModel.build(
    ssp,
    sfh={
        "type": "tsnorm",
        "*": tengri.FIXED,
        "log_peak_sfr": 1.0,
        "peak_lbt_gyr": 2.0,
        "width_gyr": 1.5,
        "skew": 0.2,
        "trunc": 2.5,
    },
    dust={"type": "two_component", "*": tengri.FIXED, "tau_diff": 0.15, "tau_bc": 0.2},
    redshift=tengri.Fixed(0.1),
)

# Evaluate
baseline_old = dict(model_old.spec.sample(jax.random.PRNGKey(0)))
baseline_recent = dict(model_recent.spec.sample(jax.random.PRNGKey(1)))
baseline_double = dict(model_double.spec.sample(jax.random.PRNGKey(2)))

out_old = model_old.predict_rest_sed(baseline_old)
out_recent = model_recent.predict_rest_sed(baseline_recent)
out_double = model_double.predict_rest_sed(baseline_double)

wave = np.asarray(out_old.wavelength)
sed_old = np.asarray(out_old.sed)
sed_recent = np.asarray(out_recent.sed)
sed_double = np.asarray(out_double.sed)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Left: Linear (optical to NIR)
mask_opt = (wave > 3000) & (wave < 3e4)
ax1.plot(
    wave[mask_opt],
    sed_old[mask_opt],
    "C0-",
    lw=2.0,
    label="Old burst only (10 Gyr ago)",
)
ax1.plot(
    wave[mask_opt],
    sed_recent[mask_opt],
    "C1-",
    lw=2.0,
    label="Recent burst only (0.3 Gyr ago)",
)
ax1.plot(
    wave[mask_opt],
    sed_double[mask_opt],
    "k--",
    lw=2.0,
    label="Double burst (combined)",
)
ax1.set_xlabel(r"Wavelength [$\AA$]", fontsize=11)
ax1.set_ylabel(r"$L_\nu$ [erg/s/Hz]", fontsize=11)
ax1.set_title("Linear Scale (Optical to NIR)", fontsize=11)
ax1.legend(fontsize=10, frameon=False)
ax1.grid(True, alpha=0.2)

# Right: Log-log (full SED)
ax2.loglog(wave, sed_old, "C0-", lw=2.0, label="Old burst only")
ax2.loglog(wave, sed_recent, "C1-", lw=2.0, label="Recent burst only")
ax2.loglog(wave, sed_double, "k--", lw=2.0, label="Double burst")
ax2.set_xlabel(r"Wavelength [$\AA$]", fontsize=11)
ax2.set_ylabel(r"$L_\nu$ [erg/s/Hz]", fontsize=11)
ax2.set_title("Log Scale (Full SED)", fontsize=11)
ax2.set_xlim(1000, 1e6)
ax2.set_ylim(1e0, 1e7)
ax2.legend(fontsize=10, frameon=False, loc="lower left")
ax2.grid(True, alpha=0.2, which="both")

fig.tight_layout()
plt.savefig("plot_sfh_double_burst.png", dpi=150, bbox_inches="tight")